# Sanitized Notebook

Outputs and Colab metadata were stripped for GitHub. The default path points to the included sample CSV; use the full local CSV for complete results.


# Lesson 4 — Payment Data Analysis: End-to-End Growth Analyst Lab
### MSc Data Science · Payment Analysis for Open Banking

---

**Dataset:** `truelayer_analytics_sample_40mb.csv`
**Scope:** 125,000 payment transactions · 142 customers · 177 banks · 14 countries · 2005–2010
**Objective:** Apply every metric from Lessons 2 & 3, then export aggregated CSVs for Metabase.

| Section | Topic |
|---------|-------|
| 0 | Setup & Data Load |
| 1 | Data Quality & EDA |
| 2 | Funnel Analysis |
| 3 | Volume, TPV & AOV |
| 4 | Failure Analysis |
| 5 | Latency Analysis |
| 6 | User Classification (First Attempt / New / Returning) |
| 7 | Frequency Tiers (Champion → Inactive) |
| 8 | RFM Analysis |
| 9 | 90-Day Forward Retention |
| 10 | Net New Active Users (Growth Accounting) |
| 11 | Metabase CSV Exports |

---
## 0 · Setup & Data Load

In [ ]:
# 0.2  Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import timedelta
import os, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})
COLORS = ['#1a73e8','#34a853','#fbbc04','#ea4335','#9c27b0','#00bcd4']
print('Libraries loaded ✓')

In [ ]:
# 0.3  Load dataset
# ← Update this path to where your file lives on Drive
FILE_PATH = '../data/sample_payments.csv'

df = pd.read_csv(
    FILE_PATH,
    parse_dates=[
        'createdat_ts','lastupdatedat_ts','initiated_at',
        'executed_at','failed_at','authorizing_at','authorized_at','settled_at'
    ]
)
print(f'Rows    : {len(df):,}')
print(f'Columns : {df.columns.tolist()}')
df.head(3)

In [ ]:
# 0.4  Normalise status column (raw data has mixed case)
df['status_norm'] = df['status'].str.strip().str.lower()

STATUS_MAP = {
    'executed':            'executed',
    'settled':             'executed',
    'failed':              'failed',
    'authorisationfailed': 'failed',
    'cancelled':           'cancelled',
    'rejected':            'rejected',
    'initiated':           'initiated',
    'initiating':          'initiated',
    'new':                 'created',
    'submitted':           'in_progress',
    'executing':           'in_progress',
    'authorizing':         'in_progress',
    'authorized':          'in_progress',
}
df['status_clean'] = df['status_norm'].map(STATUS_MAP).fillna('other')

df['is_executed'] = df['status_clean'] == 'executed'
df['is_failed']   = df['status_clean'] == 'failed'
df['is_settled']  = df['settled_at'].notna()
df['month']       = df['createdat_ts'].dt.to_period('M')

print('Status distribution:')
print(df['status_clean'].value_counts(normalize=True).mul(100).round(2))

---
## 1 · Data Quality & EDA

In [ ]:
# 1.1  Schema & null counts
print(df.dtypes)
print()
nulls = df.isnull().sum()
print('Null counts:')
print(nulls[nulls > 0].sort_values(ascending=False))

In [ ]:
# 1.2  Dataset snapshot
print(f"Date range       : {df['createdat_ts'].min().date()}  →  {df['createdat_ts'].max().date()}")
print(f"Total payments   : {len(df):,}")
print(f"Unique customers : {df['customer_id'].nunique():,}")
print(f"Unique banks     : {df['bank_id'].nunique():,}")
print(f"Unique countries : {df['country_id'].nunique():,}")
print(f"Currencies       : {sorted(df['currency'].dropna().unique().tolist())}")
print(f"Verticals        : {sorted(df['vertical'].dropna().unique().tolist())}")
print()
print('Amount stats (all payments):')
print(df['amount_in_currency'].describe().round(2))

In [ ]:
# 1.3  Payment volume over time
monthly = df.groupby('month').agg(
    payments=('id','count'),
    executed=('is_executed','sum'),
).reset_index()
monthly['month_str'] = monthly['month'].astype(str)

exec_by_month = df[df['is_executed']].groupby('month')['amount_in_currency'].sum().reset_index()
exec_by_month.columns = ['month','tpv']
monthly = monthly.merge(exec_by_month, on='month', how='left')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax1.bar(monthly['month_str'], monthly['payments'], color=COLORS[0], alpha=0.7, label='Total')
ax1.bar(monthly['month_str'], monthly['executed'], color=COLORS[1], alpha=0.9, label='Executed')
ax1.set_title('Monthly Payment Volume'); ax1.set_ylabel('Payments'); ax1.legend()
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax2.bar(monthly['month_str'], monthly['tpv'], color=COLORS[2], alpha=0.9)
ax2.set_title('Monthly TPV (Executed)'); ax2.set_ylabel('Amount')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout(); plt.show()

---
## 2 · Funnel Analysis

The payment funnel tracks drop-off between each lifecycle stage — from creation to settlement.
Each step maps to a timestamp column in the dataset.

In [ ]:
# 2.1  Build funnel from lifecycle timestamps
total       = len(df)
initiated   = df['initiated_at'].notna().sum()
authorizing = df['authorizing_at'].notna().sum()
authorized  = df['authorized_at'].notna().sum()
executed    = df['is_executed'].sum()
settled     = df['is_settled'].sum()

funnel = pd.DataFrame({
    'Stage': ['Created','Initiated','Authorizing','Authorized','Executed','Settled'],
    'Count': [total, initiated, authorizing, authorized, executed, settled]
})
funnel['drop_off_pct'] = (
    (funnel['Count'].shift(1) - funnel['Count']) / funnel['Count'].shift(1) * 100
).fillna(0).round(2)
funnel['cumulative_pct'] = (funnel['Count'] / total * 100).round(2)

print(funnel.to_string(index=False))

In [ ]:
# 2.2  Funnel chart
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(funnel['Stage'][::-1], funnel['Count'][::-1],
               color=COLORS[:len(funnel)][::-1], alpha=0.85)
for bar, row in zip(bars, funnel[::-1].itertuples()):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{row.Count:,}  ({row.cumulative_pct:.1f}%)', va='center', fontsize=9)
ax.set_title('Payment Funnel — Volume by Stage')
ax.set_xlabel('Number of Payments')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.tight_layout(); plt.show()

In [ ]:
# 2.3  E2E Conversion by Vertical and by Currency
for dim in ['vertical','currency']:
    conv = df.groupby(dim).agg(
        total=('id','count'), executed=('is_executed','sum')
    ).assign(e2e_conv=lambda x: (x['executed']/x['total']*100).round(2))
    print(f'E2E Conversion by {dim}:')
    print(conv.sort_values('e2e_conv', ascending=False))
    print()

In [ ]:
# 2.4  Monthly E2E Conversion trend
monthly_conv = df.groupby('month').agg(
    total=('id','count'), executed=('is_executed','sum')
).reset_index()
monthly_conv['e2e_conv']  = (monthly_conv['executed']/monthly_conv['total']*100).round(2)
monthly_conv['month_str'] = monthly_conv['month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly_conv['month_str'], monthly_conv['e2e_conv'],
        marker='o', color=COLORS[0], linewidth=2)
avg = monthly_conv['e2e_conv'].mean()
ax.axhline(avg, color='grey', linestyle='--', linewidth=1, label=f'Avg: {avg:.1f}%')
ax.set_title('Monthly E2E Conversion Rate (%)'); ax.set_ylabel('Conversion %')
ax.set_ylim(0, 100); ax.legend()
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout(); plt.show()

---
## 3 · Volume, TPV & AOV

> **AOV = Total Amount Settled / Number of Settled Payments**
> **TPV** is the North Star. AOV and PPU are the two lenses to decompose it.

In [ ]:
# 3.1  Overall KPIs
exec_df = df[df['is_executed']].copy()

tpv      = exec_df['amount_in_currency'].sum()
n_exec   = len(exec_df)
aov      = tpv / n_exec
n_users  = exec_df['customer_id'].nunique()
ppu      = n_exec / n_users

print('=== CORE KPIs (Executed Payments) ===')
print(f'TPV           : {tpv:,.2f}')
print(f'Transactions  : {n_exec:,}')
print(f'AOV           : {aov:.2f}')
print(f'Active users  : {n_users:,}')
print(f'PPU           : {ppu:.2f}')
print(f'LTV proxy     : {aov * ppu:.2f}  (AOV × PPU)')

In [ ]:
# 3.2  AOV by Vertical and Currency
for dim in ['vertical','currency']:
    g = exec_df.groupby(dim).agg(
        txns=('id','count'), tpv=('amount_in_currency','sum')
    ).assign(aov=lambda x: (x['tpv']/x['txns']).round(2))
    print(f'AOV by {dim}:')
    print(g.sort_values('aov', ascending=False))
    print()

In [ ]:
# 3.3  Monthly AOV trend
monthly_aov = exec_df.groupby('month').agg(
    tpv=('amount_in_currency','sum'), txns=('id','count')
).reset_index()
monthly_aov['aov']       = (monthly_aov['tpv']/monthly_aov['txns']).round(2)
monthly_aov['month_str'] = monthly_aov['month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly_aov['month_str'], monthly_aov['aov'],
        marker='s', color=COLORS[2], linewidth=2)
ax.set_title('Monthly AOV'); ax.set_ylabel('Avg Order Value')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# 3.4  Top 10 banks by TPV
top_banks = exec_df.groupby('bank_id').agg(
    txns=('id','count'), tpv=('amount_in_currency','sum')
).sort_values('tpv', ascending=False).head(10).reset_index()
top_banks['bank_short'] = top_banks['bank_id'].str[:12] + '...'
print('Top 10 Banks by TPV:')
print(top_banks[['bank_short','txns','tpv']].to_string(index=False))

---
## 4 · Failure Analysis

Failures are grouped by **reason** (why) and **stage** (where in the lifecycle).
- Failures at `authorizing` stage → likely bank-side rejection
- Failures at `initiated` stage → likely user abandonment or timeout

In [ ]:
# 4.1  Failure overview
fail_df = df[df['is_failed']].copy()
print(f"Failure rate     : {len(fail_df)/len(df)*100:.2f}%  ({len(fail_df):,} payments)")
print()
print('Failure Reasons (top 15):')
print(fail_df['failure_reason'].value_counts().head(15))
print()
print('Failure Stage:')
print(fail_df['failure_stage'].value_counts())

In [ ]:
# 4.2  Failure rate by Vertical and Bank
print('Failure rate by Vertical:')
fv = df.groupby('vertical').agg(total=('id','count'), failed=('is_failed','sum'))
fv['failure_rate'] = (fv['failed']/fv['total']*100).round(2)
print(fv.sort_values('failure_rate', ascending=False))
print()

print('Top 10 Banks by Failure Rate (min 50 txns):')
fb = df.groupby('bank_id').agg(total=('id','count'), failed=('is_failed','sum'))
fb = fb[fb['total'] >= 50].copy()
fb['failure_rate'] = (fb['failed']/fb['total']*100).round(2)
fb['bank_short']   = fb.index.str[:14] + '...'
print(fb.sort_values('failure_rate', ascending=False).head(10)[
      ['bank_short','total','failed','failure_rate']].to_string())

In [ ]:
# 4.3  Failure reason bar chart
reason_counts = fail_df['failure_reason'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(reason_counts.index[::-1], reason_counts.values[::-1], color=COLORS[3])
ax.set_title('Top Failure Reasons'); ax.set_xlabel('Count')
plt.tight_layout(); plt.show()

---
## 5 · Latency Analysis

Latency measures the time (in seconds) between each lifecycle stage.
High latency at a specific step may indicate bank-side delays, connectivity issues, or authorisation bottlenecks.

In [ ]:
# 5.1  Compute latencies for executed payments
exec_df = df[df['is_executed']].copy()
exec_df['lat_create_to_initiate']    = (exec_df['initiated_at']  - exec_df['createdat_ts']).dt.total_seconds()
exec_df['lat_initiate_to_authorize'] = (exec_df['authorized_at'] - exec_df['initiated_at']).dt.total_seconds()
exec_df['lat_authorize_to_execute']  = (exec_df['executed_at']   - exec_df['authorized_at']).dt.total_seconds()
exec_df['lat_e2e']                   = (exec_df['executed_at']   - exec_df['createdat_ts']).dt.total_seconds()
exec_df['lat_execute_to_settle']     = (exec_df['settled_at']    - exec_df['executed_at']).dt.total_seconds()

lat_cols = ['lat_create_to_initiate','lat_initiate_to_authorize',
            'lat_authorize_to_execute','lat_e2e','lat_execute_to_settle']
print('Latency stats (seconds):')
print(exec_df[lat_cols].describe().round(1))

In [ ]:
# 5.2  E2E latency distribution (capped at 300s for readability)
lat_clean = exec_df['lat_e2e'].dropna()
lat_clean = lat_clean[(lat_clean > 0) & (lat_clean <= 300)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lat_clean, bins=60, color=COLORS[0], edgecolor='white', alpha=0.85)
ax.axvline(lat_clean.median(), color=COLORS[3], linestyle='--',
           label=f'Median: {lat_clean.median():.1f}s')
ax.set_title('E2E Latency Distribution (Create → Execute, capped 300s)')
ax.set_xlabel('Seconds'); ax.set_ylabel('Count'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# 5.3  Average E2E latency by bank (top 15 slowest, min 50 txns)
bank_latency = exec_df.groupby('bank_id').agg(
    txns=('id','count'),
    median_e2e=('lat_e2e','median'),
    p95_e2e=('lat_e2e', lambda x: x.quantile(0.95))
).query('txns >= 50').sort_values('median_e2e', ascending=False).head(15).reset_index()
bank_latency['bank_short'] = bank_latency['bank_id'].str[:14] + '...'
print('Slowest Banks by Median E2E Latency:')
print(bank_latency[['bank_short','txns','median_e2e','p95_e2e']].to_string(index=False))

---
## 6 · User Classification

Each payment event is labelled with the user's lifecycle stage at the time of that payment:

| Stage | Definition |
|-------|------------|
| **First Attempt** | The very first payment attempt ever recorded for that user |
| **New User** | Every attempt from the first to the first *successful* payment (inclusive) |
| **Returning User** | Every attempt *after* the first successful payment |

This classification lets you measure conversion rate and AOV separately for each cohort.

In [ ]:
# 6.1  Label each payment with the user's lifecycle stage
df_sorted = df.sort_values(['customer_id','createdat_ts']).copy()
df_sorted['attempt_rank'] = df_sorted.groupby('customer_id').cumcount() + 1

# First successful payment timestamp per user
first_success = (
    df_sorted[df_sorted['is_executed']]
    .groupby('customer_id')['createdat_ts']
    .min()
    .rename('first_success_ts')
)
df_sorted = df_sorted.join(first_success, on='customer_id')

def classify_stage(row):
    if row['attempt_rank'] == 1:
        return 'first_attempt'
    if pd.isna(row['first_success_ts']):
        return 'new_user'
    if row['createdat_ts'] <= row['first_success_ts']:
        return 'new_user'
    return 'returning_user'

df_sorted['user_stage'] = df_sorted.apply(classify_stage, axis=1)

print('Payment count by user stage:')
print(df_sorted['user_stage'].value_counts())
print()
print('E2E Conversion by user stage:')
print(df_sorted.groupby('user_stage').agg(
    payments=('id','count'), executed=('is_executed','sum')
).assign(e2e_conv=lambda x: (x['executed']/x['payments']*100).round(2)))

---
## 7 · Frequency Tiers

Activity-based segmentation groups users by total executed payments (lifetime).

| Tier | Threshold | Description |
|------|-----------|-------------|
| Champion | 20+ | Power users, highest LTV |
| Super | 10–19 | High-value regulars |
| Engaged | 5–9 | Consistent but moderate |
| Light | 2–4 | Occasional |
| Inactive | 1 | Single transaction only |

> Adjust thresholds to match the actual distribution in your dataset.

In [ ]:
# 7.1  Compute frequency tier per user
user_freq = (
    df[df['is_executed']]
    .groupby('customer_id')
    .agg(payments=('id','count'), tpv=('amount_in_currency','sum'))
    .reset_index()
)

def freq_tier(n):
    if n >= 20: return 'Champion'
    if n >= 10: return 'Super'
    if n >= 5:  return 'Engaged'
    if n >= 2:  return 'Light'
    return 'Inactive'

user_freq['tier'] = user_freq['payments'].apply(freq_tier)

tier_order = ['Champion','Super','Engaged','Light','Inactive']
tier_summary = (
    user_freq.groupby('tier')
    .agg(users=('customer_id','count'), total_tpv=('tpv','sum'))
    .reindex(tier_order)
    .reset_index()
)
tier_summary['pct_users'] = (tier_summary['users']/tier_summary['users'].sum()*100).round(1)
tier_summary['pct_tpv']   = (tier_summary['total_tpv']/tier_summary['total_tpv'].sum()*100).round(1)
print('Frequency Tier Pareto:')
print(tier_summary.to_string(index=False))

In [ ]:
# 7.2  Pareto chart — % users vs % TPV
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(tier_summary)); w = 0.35
bars1 = ax.bar([i-w/2 for i in x], tier_summary['pct_users'], w, label='% Users', color=COLORS[0], alpha=0.85)
bars2 = ax.bar([i+w/2 for i in x], tier_summary['pct_tpv'],   w, label='% TPV',   color=COLORS[2], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(tier_summary['tier'])
ax.set_ylabel('Percentage (%)'); ax.set_title('Frequency Tier: % Users vs % TPV'); ax.legend()
for b in list(bars1)+list(bars2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{b.get_height():.1f}%', ha='center', fontsize=8)
plt.tight_layout(); plt.show()

---
## 8 · RFM Analysis

| Dimension | Definition | Score |
|-----------|------------|-------|
| **Recency** | Days since last executed payment | Lower = better → higher score |
| **Frequency** | Number of executed payments | Higher = better |
| **Monetary** | Total settled amount | Higher = better |

Each dimension is scored 1–5 using quintiles. Scores are concatenated into an RFM code (e.g. `545`).

In [ ]:
# 8.1  Compute R, F, M per user
SNAPSHOT_DATE = df['createdat_ts'].max()

rfm = (
    df[df['is_executed']]
    .groupby('customer_id')
    .agg(
        last_payment=('createdat_ts','max'),
        frequency=('id','count'),
        monetary=('amount_in_currency','sum')
    )
    .reset_index()
)
rfm['recency_days'] = (SNAPSHOT_DATE - rfm['last_payment']).dt.days
print('RFM base stats:')
print(rfm[['recency_days','frequency','monetary']].describe().round(1))

In [ ]:
# 8.2  Score each dimension 1–5 using quintiles
def quintile_score(series, ascending=True):
    labels = [5,4,3,2,1] if ascending else [1,2,3,4,5]
    return pd.qcut(series, q=5, labels=labels, duplicates='drop').astype(int)

rfm['R'] = quintile_score(rfm['recency_days'], ascending=True)   # lower recency → better → score 5
rfm['F'] = quintile_score(rfm['frequency'],    ascending=False)
rfm['M'] = quintile_score(rfm['monetary'],     ascending=False)
rfm['RFM_score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['RFM_total'] = rfm['R'] + rfm['F'] + rfm['M']

def rfm_segment(row):
    r, f, m = row['R'], row['F'], row['M']
    if r >= 4 and f >= 4 and m >= 4: return 'Champions'
    if r >= 3 and f >= 3:            return 'Loyal'
    if r >= 4 and f <= 2:            return 'Promising'
    if r <= 2 and f >= 3:            return 'At Risk'
    if r <= 2 and f <= 2 and m >= 3: return 'Needs Attention'
    if r == 1 and f == 1:            return 'Lost'
    return 'Hibernating'

rfm['segment'] = rfm.apply(rfm_segment, axis=1)
print(rfm['segment'].value_counts())

In [ ]:
# 8.3  Segment summary table
seg_summary = rfm.groupby('segment').agg(
    users=('customer_id','count'),
    avg_recency=('recency_days','mean'),
    avg_frequency=('frequency','mean'),
    avg_monetary=('monetary','mean'),
    total_tpv=('monetary','sum')
).round(1).sort_values('total_tpv', ascending=False)
print(seg_summary.to_string())

In [ ]:
# 8.4  RFM Bubble chart — Recency vs Frequency, bubble = TPV
fig, ax = plt.subplots(figsize=(11, 6))
for i, (seg, row) in enumerate(seg_summary.iterrows()):
    ax.scatter(
        row['avg_recency'], row['avg_frequency'],
        s=row['total_tpv']/seg_summary['total_tpv'].max()*3000,
        color=COLORS[i % len(COLORS)], alpha=0.75, edgecolors='white', linewidth=1.5
    )
    ax.annotate(seg, (row['avg_recency'], row['avg_frequency']),
                textcoords='offset points', xytext=(8,3), fontsize=9)
ax.set_xlabel('Avg Recency (days — lower = more recent)')
ax.set_ylabel('Avg Frequency')
ax.set_title('RFM Segments — bubble size = Total TPV')
plt.tight_layout(); plt.show()

---
## 9 · 90-Day Forward-Looking Retention

For each executed payment on day X: did the user come back within the **next 90 days**?

- Users whose 90-day window extends beyond the snapshot date are **censored** and excluded from the denominator.
- This avoids systematically underestimating retention for recent events.

In [ ]:
# 9.1  Build event-level retention table
SNAPSHOT     = df['createdat_ts'].max()
WINDOW_DAYS  = 90

events = df[df['is_executed']][['customer_id','createdat_ts']].copy()
events = events.rename(columns={'createdat_ts':'payment_date'})
events = events.sort_values(['customer_id','payment_date'])
events['next_payment'] = events.groupby('customer_id')['payment_date'].shift(-1)
events['window_end']   = events['payment_date'] + timedelta(days=WINDOW_DAYS)
events['censored']     = events['window_end'] > SNAPSHOT
events['retained']     = (
    events['next_payment'].notna() &
    (events['next_payment'] <= events['window_end'])
)

eligible = events[~events['censored']]
rate = eligible['retained'].sum() / len(eligible) * 100

print(f'Total events       : {len(events):,}')
print(f'Censored           : {events["censored"].sum():,}')
print(f'Eligible           : {len(eligible):,}')
print(f'Retained           : {int(eligible["retained"].sum()):,}')
print(f'90-Day Retention   : {rate:.2f}%')

In [ ]:
# 9.2  Monthly retention rate trend
events['month'] = events['payment_date'].dt.to_period('M')
monthly_ret = (
    events[~events['censored']]
    .groupby('month')
    .agg(eligible=('customer_id','count'), retained=('retained','sum'))
    .reset_index()
)
monthly_ret['retention_rate'] = (monthly_ret['retained']/monthly_ret['eligible']*100).round(2)
monthly_ret['month_str'] = monthly_ret['month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly_ret['month_str'], monthly_ret['retention_rate'],
        marker='o', color=COLORS[1], linewidth=2)
ax.axhline(monthly_ret['retention_rate'].mean(), linestyle='--', color='grey',
           label=f"Avg: {monthly_ret['retention_rate'].mean():.1f}%")
ax.set_title('Monthly 90-Day Retention Rate (forward-looking, eligible events only)')
ax.set_ylabel('Retention %'); ax.set_ylim(0, 100); ax.legend()
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout(); plt.show()

---
## 10 · Net New Active Users (Growth Accounting)

$$\Delta MAU = \text{New} + \text{Resurrected} - \text{Churned}$$

| Component | Definition |
|-----------|------------|
| **New** | First executed payment ever this month |
| **Retained** | Active last month and this month |
| **Resurrected** | Inactive last month, active this month, not first-ever |
| **Churned** | Active last month, inactive this month |

In [ ]:
# 10.1  Build monthly growth accounting table
user_months = (
    df[df['is_executed']]
    .assign(month=lambda x: x['createdat_ts'].dt.to_period('M'))
    .groupby(['customer_id','month'])
    .size()
    .reset_index(name='payments')
)

first_month = user_months.groupby('customer_id')['month'].min().rename('first_month')
user_months = user_months.join(first_month, on='customer_id')

all_months = sorted(user_months['month'].unique())
result = []

for i, month in enumerate(all_months):
    curr_users = set(user_months[user_months['month'] == month]['customer_id'])
    prev_users = set(user_months[user_months['month'] == all_months[i-1]]['customer_id']) if i > 0 else set()
    new_users   = {u for u in curr_users if first_month[u] == month}
    retained    = curr_users & prev_users - new_users
    resurrected = curr_users - prev_users - new_users
    churned     = prev_users - curr_users
    result.append({
        'month': str(month),
        'MAU': len(curr_users),
        'new': len(new_users),
        'retained': len(retained),
        'resurrected': len(resurrected),
        'churned': len(churned),
        'net_change': len(new_users) + len(resurrected) - len(churned)
    })

growth = pd.DataFrame(result)
print(growth.to_string(index=False))

In [ ]:
# 10.2  Growth accounting stacked bar
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(growth['month'], growth['new'],         color=COLORS[1], label='New')
ax.bar(growth['month'], growth['resurrected'],
       bottom=growth['new'],                   color=COLORS[0], label='Resurrected')
ax.bar(growth['month'], -growth['churned'],    color=COLORS[3], label='Churned')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Monthly Growth Accounting: New + Resurrected − Churned')
ax.set_ylabel('Users'); ax.legend()
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout(); plt.show()

---
## 11 · Metabase CSV Exports

The cells below generate **8 pre-aggregated CSV files** ready to import into Metabase.

| File | Use in Metabase |
|------|----------------|
| `monthly_kpi_summary.csv` | KPI trend lines (TPV, AOV, conversion, retention) |
| `payment_funnel.csv` | Funnel bar / waterfall chart |
| `vertical_performance.csv` | Vertical comparison table |
| `failure_summary.csv` | Failure reason breakdown |
| `user_rfm_segments.csv` | RFM scatter / segment pie |
| `user_frequency_tiers.csv` | Tier distribution |
| `growth_accounting.csv` | MAU & growth waterfall |
| `bank_quality_scorecard.csv` | Bank-level conversion and failure |

**How to import:** Metabase v49+ → *New → Upload CSV* or *Admin → Databases → Add CSV*.

In [ ]:
# 11.0  Create output folder
OUT = '/content/drive/MyDrive/PaymentAnalysisLab/metabase_exports/'
os.makedirs(OUT, exist_ok=True)
print(f'Export folder: {OUT}')

In [ ]:
# 11.1  Monthly KPI Summary
kpi = monthly_conv.merge(
    monthly_aov[['month','tpv','aov']], on='month'
).merge(
    monthly_ret[['month','retention_rate']], on='month', how='left'
)[['month_str','total','executed','e2e_conv','tpv','aov','retention_rate']]
kpi.columns = ['month','total_payments','executed_payments',
               'e2e_conversion_pct','tpv','aov','retention_rate_90d']
kpi.to_csv(OUT+'monthly_kpi_summary.csv', index=False)
print(f'Saved {len(kpi)} rows → monthly_kpi_summary.csv')
kpi.tail(5)

In [ ]:
# 11.2  Payment Funnel
funnel.to_csv(OUT+'payment_funnel.csv', index=False)
print(f'Saved {len(funnel)} rows → payment_funnel.csv')

In [ ]:
# 11.3  Vertical Performance
vperf = df.groupby('vertical').agg(
    total_payments=('id','count'),
    executed=('is_executed','sum'),
    failed=('is_failed','sum'),
).reset_index()
vperf['tpv'] = df[df['is_executed']].groupby('vertical')['amount_in_currency'].sum().values
vperf['aov'] = df[df['is_executed']].groupby('vertical')['amount_in_currency'].mean().values
vperf['e2e_conversion_pct'] = (vperf['executed']/vperf['total_payments']*100).round(2)
vperf['failure_rate_pct']   = (vperf['failed']/vperf['total_payments']*100).round(2)
vperf.to_csv(OUT+'vertical_performance.csv', index=False)
print(f'Saved {len(vperf)} rows → vertical_performance.csv')
print(vperf)

In [ ]:
# 11.4  Failure Summary
fail_export = (
    fail_df.groupby(['failure_reason','failure_stage'])
    .agg(count=('id','count'))
    .reset_index()
    .sort_values('count', ascending=False)
)
fail_export['pct_of_failures'] = (fail_export['count']/len(fail_df)*100).round(2)
fail_export.to_csv(OUT+'failure_summary.csv', index=False)
print(f'Saved {len(fail_export)} rows → failure_summary.csv')

In [ ]:
# 11.5  User RFM Segments
rfm_export = rfm[['customer_id','recency_days','frequency','monetary',
                   'R','F','M','RFM_score','RFM_total','segment']].copy()
rfm_export.to_csv(OUT+'user_rfm_segments.csv', index=False)
print(f'Saved {len(rfm_export)} rows → user_rfm_segments.csv')

In [ ]:
# 11.6  User Frequency Tiers
user_freq[['customer_id','payments','tpv','tier']].to_csv(
    OUT+'user_frequency_tiers.csv', index=False)
print(f'Saved {len(user_freq)} rows → user_frequency_tiers.csv')

In [ ]:
# 11.7  Growth Accounting
growth.to_csv(OUT+'growth_accounting.csv', index=False)
print(f'Saved {len(growth)} rows → growth_accounting.csv')

In [ ]:
# 11.8  Bank Quality Scorecard (min 50 txns)
bank_sc = df.groupby('bank_id').agg(
    total_payments=('id','count'),
    executed=('is_executed','sum'),
    failed=('is_failed','sum'),
).reset_index()
bank_sc = bank_sc[bank_sc['total_payments'] >= 50].copy()

bank_tpv = df[df['is_executed']].groupby('bank_id')['amount_in_currency'].sum().rename('tpv')
bank_aov = df[df['is_executed']].groupby('bank_id')['amount_in_currency'].mean().rename('aov')
bank_sc = bank_sc.join(bank_tpv, on='bank_id').join(bank_aov, on='bank_id')
bank_sc['e2e_conversion_pct'] = (bank_sc['executed']/bank_sc['total_payments']*100).round(2)
bank_sc['failure_rate_pct']   = (bank_sc['failed']/bank_sc['total_payments']*100).round(2)
bank_sc = bank_sc.round(2)
bank_sc.to_csv(OUT+'bank_quality_scorecard.csv', index=False)
print(f'Saved {len(bank_sc)} rows → bank_quality_scorecard.csv')

In [ ]:
# 11.9  Export summary
print('=== METABASE EXPORT COMPLETE ===')
for fname in [
    'monthly_kpi_summary.csv','payment_funnel.csv','vertical_performance.csv',
    'failure_summary.csv','user_rfm_segments.csv','user_frequency_tiers.csv',
    'growth_accounting.csv','bank_quality_scorecard.csv'
]:
    fpath = OUT + fname
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath)/1024
        rows    = sum(1 for _ in open(fpath)) - 1
        print(f'  ✓  {fname:<42} {rows:>5} rows  {size_kb:>6.1f} KB')
print()
print('Import each file into Metabase: New → Upload CSV')

---
## Discussion Questions

1. **Conversion** — Which vertical has the lowest E2E conversion? What hypotheses can you form about the cause?
2. **Failure** — Is failure rate uniform across banks? What does a high bank-level rate tell you about root cause?
3. **AOV** — Does AOV vary meaningfully by currency? What would you expect and why?
4. **RFM** — Which segment drives the most TPV? Is it the same segment with the most users?
5. **Retention** — Plot 90-day retention and TPV on the same time axis. Describe the relationship.
6. **Growth Accounting** — Is the user base growing or shrinking in the final quarter? Decompose the trend.
7. **Metabase** — After importing all exports, build a dashboard answering: *"How healthy is this payment business?"* Justify your chart choices.

---
*Lesson 4 · Payment Analysis for Open Banking · MSc Data Science & Economics*